In [ ]:
import copy
from collections import deque

ROWS = 9
COLS = 9
DIGITS = set(range(1, 10))

def get_peers():
    """Precomputes the set of neighbors (peers) for every cell."""
    peers = {}
    for r in range(ROWS):
        for c in range(COLS):
            cell_peers = set()
            # Row and column peers
            for k in range(9):
                if k != c:
                    cell_peers.add((r, k))
                if k != r:
                    cell_peers.add((k, c))
            # 3x3 Subgrid peers
            box_r, box_c = 3 * (r // 3), 3 * (c // 3)
            for i in range(box_r, box_r + 3):
                for j in range(box_c, box_c + 3):
                    if (i, j) != (r, c):
                        cell_peers.add((i, j))
            peers[(r, c)] = cell_peers
    return peers

PEERS = get_peers()


def init_domains(grid):
    """Initializes domains from the input grid (0 represents empty cells)."""
    domains = {}
    for r in range(ROWS):
        for c in range(COLS):
            val = grid[r][c]
            if val != 0:
                domains[(r, c)] = {val}
            else:
                domains[(r, c)] = set(DIGITS)
    return domains


def ac3(domains, queue=None):
    """
    AC-3 algorithm for arc consistency.
    Enforces binary constraint: cell_i != cell_j.
    """
    if queue is None:
        queue = deque()
        for var in domains:
            for neighbor in PEERS[var]:
                queue.append((var, neighbor))

    while queue:
        xi, xj = queue.popleft()
        if revise(domains, xi, xj):
            # If domain is empty, CSP has reached an invalid state
            if len(domains[xi]) == 0:
                return False
            # Propagate changes to xi's other neighbors
            for xk in PEERS[xi]:
                if xk != xj:
                    queue.append((xk, xi))
    return True


def revise(domains, xi, xj):
    """
    Removes values from domains[xi] if they have no valid match in domains[xj].
    For Sudoku, if xj is fixed to one value, xi cannot take that value.
    """
    revised = False
    if len(domains[xj]) == 1:
        fixed_val = next(iter(domains[xj]))
        if fixed_val in domains[xi]:
            domains[xi].remove(fixed_val)
            revised = True
    return revised


def is_solved(domains):
    """Check if all variables have been assigned exactly one value."""
    return all(len(domains[var]) == 1 for var in domains)


def select_unassigned_variable_mrv(domains):
    """
    MRV (Minimum Remaining Values) heuristic:
    Selects the cell with the smallest domain size (> 1).
    """
    unassigned = [var for var in domains if len(domains[var]) > 1]
    return min(unassigned, key=lambda var: len(domains[var]))


def backtrack(domains):
    """Backtracking search with constraint propagation (AC-3)."""
    if is_solved(domains):
        return domains

    # Pick next variable using MRV
    var = select_unassigned_variable_mrv(domains)

    for value in sorted(domains[var]):
        domains_copy = copy.deepcopy(domains)
        domains_copy[var] = {value}

        # Initialize queue with arcs pointing to the modified variable
        queue = deque((neighbor, var) for neighbor in PEERS[var])
        
        # Enforce consistency after assignment
        if ac3(domains_copy, queue):
            result = backtrack(domains_copy)
            if result is not None:
                return result

    return None


def solve_sudoku(grid):
    """Main solver function."""
    domains = init_domains(grid)
    
    if not ac3(domains):
        return None

    solved_domains = backtrack(domains)
    if solved_domains is None:
        return None

    solution = [[0] * COLS for _ in range(ROWS)]
    for (r, c), values in solved_domains.items():
        solution[r][c] = next(iter(values))
    return solution


def print_grid(grid):
    """Utility to print the board neatly."""
    for r in range(9):
        if r % 3 == 0 and r != 0:
            print("-" * 21)
        row_str = []
        for c in range(9):
            if c % 3 == 0 and c != 0:
                row_str.append("|")
            val = grid[r][c]
            row_str.append(str(val) if val != 0 else ".")
        print(" ".join(row_str))

In [ ]:
if __name__ == "__main__":
    puzzle = [
        [5, 3, 0, 0, 7, 0, 0, 0, 0],
        [6, 0, 0, 1, 9, 5, 0, 0, 0],
        [0, 9, 8, 0, 0, 0, 0, 6, 0],
        [8, 0, 0, 0, 6, 0, 0, 0, 3],
        [4, 0, 0, 8, 0, 3, 0, 0, 1],
        [7, 0, 0, 0, 2, 0, 0, 0, 6],
        [0, 6, 0, 0, 0, 0, 2, 8, 0],
        [0, 0, 0, 4, 1, 9, 0, 0, 5],
        [0, 0, 0, 0, 8, 0, 0, 7, 9]
    ]

    print("Input Puzzle:")
    print_grid(puzzle)
    print("\nSolving...\n")

    solution = solve_sudoku(puzzle)

    if solution:
        print("Solved Puzzle:")
        print_grid(solution)
    else:
        print("No solution exists.")

Input Puzzle:
5 3 . | . 7 . | . . .
6 . . | 1 9 5 | . . .
. 9 8 | . . . | . 6 .
---------------------
8 . . | . 6 . | . . 3
4 . . | 8 . 3 | . . 1
7 . . | . 2 . | . . 6
---------------------
. 6 . | . . . | 2 8 .
. . . | 4 1 9 | . . 5
. . . | . 8 . | . 7 9

Solving...

Solved Puzzle:
5 3 4 | 6 7 8 | 9 1 2
6 7 2 | 1 9 5 | 3 4 8
1 9 8 | 3 4 2 | 5 6 7
---------------------
8 5 9 | 7 6 1 | 4 2 3
4 2 6 | 8 5 3 | 7 9 1
7 1 3 | 9 2 4 | 8 5 6
---------------------
9 6 1 | 5 3 7 | 2 8 4
2 8 7 | 4 1 9 | 6 3 5
3 4 5 | 2 8 6 | 1 7 9
